# Exploration / Scratch

Free-form notebook for poking at the data. Use this to:
- Spot-check parse output on specific PDFs
- Run ad-hoc DuckDB queries
- Try ideas before committing them to source modules

**This notebook is intentionally minimal — copy cells from here into `01_task1_ingest.ipynb` etc. when you find something useful.**

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2
from src.common.db import get_connection
import pandas as pd
pd.set_option("display.max_colwidth", 200)

# Look at TF-IDF's pick (July 16)
with get_connection(read_only=True) as con:
    df_tfidf = con.execute("""
        SELECT start_time, end_time, end_depth_md, main_activity, sub_activity, state, remark
        FROM operations o
        JOIN reports r ON o.pdf_path = r.pdf_path
        WHERE r.pdf_filename = '15_9_F_12_2007_07_16.pdf'
        ORDER BY start_time
    """).df()
print("=== TF-IDF picked: July 16 ===")
display(df_tfidf)

INFO     NumExpr defaulting to 12 threads.

=== TF-IDF picked: July 16 ===


,start_time,end_time,end_depth_md,main_activity,sub_activity,state,remark
0,00:30,01:00,1357.0,drilling,casing,ok,Held tool box talk prior to R/D cmt hose. R/D same cmt hose and control lines for operating the cmt head.
1,01:00,01:30,1357.0,drilling,casing,ok,"Set down 2 MT on CART and marked DP on drill floor. Turned string 5 right hand turn and released CART. Pulled 1.5m above 18 3/4"" Subsea well head."
2,01:30,02:30,130.0,drilling,casing,ok,"Washed 18 3/4"" Subsea well head area with SW at 5000 lpm, using the landing string as washing tool. Inspected 18 3/4"" Subsea well head, found both pad eyes vi sible."
3,02:30,03:45,130.0,drilling,casing,ok,"Held tool box talk prior to L/D cmt head, L/D cmt head."
4,03:45,04:15,130.0,drilling,casing,ok,Dropped two Halliburton sponge balls and flush string using seawater at maximum rate.
5,04:15,06:00,0.0,drilling,casing,ok,Pull out of water w/ landing string and CART. Measured landing string.
6,06:00,07:45,0.0,drilling,casing,ok,"Continued L/O landing string. Held tool box talk with day shift prior to L/O CART. L/O 18 3/4"" CART. B/O subs from FAC tool and L/O same."
7,07:45,08:15,0.0,drilling,trip,ok,"Cleared rig floor and excess equipment in preparation for L/O 26"" BHA."
8,08:15,09:30,0.0,drilling,trip,ok,"Held tool box talk prior to L/O 26"" BHA. L/O 26"" bit and MWD power puls. Broke dog collar links."
9,09:30,10:00,0.0,interruption,maintain,ok,Replaced broken dog collar links.


In [2]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.common.db import get_connection
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 240)

with get_connection(read_only=True) as con:
    # ---- 1. Tables and row counts ----
    print("=" * 70)
    print("1. TABLES & ROW COUNTS")
    print("=" * 70)
    tables = ["reports", "operations", "drilling_fluid", "pore_pressure",
              "survey_station", "lithology", "gas_reading", "parse_errors"]
    for t in tables:
        n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        print(f"  {t:20s} {n:>8,}")

    # ---- 2. reports — header field population rate ----
    print("\n" + "=" * 70)
    print("2. REPORTS — populated count per column (out of total reports)")
    print("=" * 70)
    total_reports = con.execute("SELECT COUNT(*) FROM reports").fetchone()[0]
    cols = con.execute("PRAGMA table_info('reports')").df()["name"].tolist()
    rows = []
    for c in cols:
        n = con.execute(f"SELECT COUNT({c}) FROM reports").fetchone()[0]
        rows.append({"column": c, "populated": n, "pct": round(100 * n / total_reports, 1)})
    print(pd.DataFrame(rows).to_string(index=False))

    # ---- 3. reports — well distribution ----
    print("\n" + "=" * 70)
    print("3. REPORTS — distribution by well_family")
    print("=" * 70)
    print(con.execute("""
        SELECT well_family, COUNT(*) AS n,
               MIN(report_date) AS earliest,
               MAX(report_date) AS latest,
               COUNT(DISTINCT operator) AS n_operators,
               COUNT(DISTINCT rig_name) AS n_rigs
        FROM reports
        GROUP BY well_family
        ORDER BY n DESC
    """).df().to_string(index=False))

    # ---- 4. reports — sample of populated header fields ----
    print("\n" + "=" * 70)
    print("4. REPORTS — sample of 3 random rows (key header fields)")
    print("=" * 70)
    print(con.execute("""
        SELECT pdf_filename, wellbore_id, period_start, status, operator,
               rig_name, spud_date, depth_md, hole_dia_in, parse_quality
        FROM reports
        ORDER BY RANDOM()
        LIMIT 3
    """).df().to_string(index=False))

    # ---- 5. operations — column population ----
    print("\n" + "=" * 70)
    print("5. OPERATIONS — populated count per column")
    print("=" * 70)
    total_ops = con.execute("SELECT COUNT(*) FROM operations").fetchone()[0]
    op_cols = con.execute("PRAGMA table_info('operations')").df()["name"].tolist()
    rows = []
    for c in op_cols:
        n = con.execute(f"SELECT COUNT({c}) FROM operations").fetchone()[0]
        rows.append({"column": c, "populated": n, "pct": round(100 * n / total_ops, 1)})
    print(pd.DataFrame(rows).to_string(index=False))

    # ---- 6. operations — top activities & states ----
    print("\n" + "=" * 70)
    print("6. OPERATIONS — top main_activity / sub_activity / state combos")
    print("=" * 70)
    print(con.execute("""
        SELECT main_activity, sub_activity, state, COUNT(*) AS n
        FROM operations
        WHERE main_activity IS NOT NULL
        GROUP BY 1, 2, 3
        ORDER BY n DESC
        LIMIT 15
    """).df().to_string(index=False))

    # ---- 7. operations — sample row inspection ----
    print("\n" + "=" * 70)
    print("7. OPERATIONS — 3 random rows with full content")
    print("=" * 70)
    print(con.execute("""
        SELECT op_id, well_family, report_date, start_time, end_time,
               end_depth_md, main_activity, sub_activity, state,
               SUBSTRING(remark, 1, 80) AS remark_preview,
               SUBSTRING(op_text, 1, 80) AS op_text_preview
        FROM operations
        WHERE op_text IS NOT NULL
        ORDER BY RANDOM()
        LIMIT 3
    """).df().to_string(index=False))

    # ---- 8. drilling_fluid sample ----
    print("\n" + "=" * 70)
    print("8. DRILLING_FLUID — 3 sample rows")
    print("=" * 70)
    print(con.execute("""
        SELECT well_family, sample_time, sample_point, sample_depth_md,
               fluid_type, fluid_density_gcm3, plastic_visc_mpas, yield_point_pa
        FROM drilling_fluid
        WHERE fluid_type IS NOT NULL
        ORDER BY RANDOM()
        LIMIT 3
    """).df().to_string(index=False))

    # ---- 9. survey_station sample ----
    print("\n" + "=" * 70)
    print("9. SURVEY_STATION — 3 sample rows")
    print("=" * 70)
    print(con.execute("""
        SELECT well_family, depth_md, depth_tvd, inclination_deg, azimuth_deg
        FROM survey_station
        ORDER BY RANDOM()
        LIMIT 3
    """).df().to_string(index=False))

    # ---- 10. lithology sample ----
    print("\n" + "=" * 70)
    print("10. LITHOLOGY — 3 sample rows")
    print("=" * 70)
    print(con.execute("""
        SELECT well_family, start_depth_md, end_depth_md,
               SUBSTRING(lithology_description, 1, 80) AS description
        FROM lithology
        ORDER BY RANDOM()
        LIMIT 3
    """).df().to_string(index=False))

    # ---- 11. pore_pressure sample ----
    print("\n" + "=" * 70)
    print("11. PORE_PRESSURE — 3 sample rows")
    print("=" * 70)
    print(con.execute("""
        SELECT well_family, sample_time, depth_md, equ_mud_weight_gcm3, reading_type
        FROM pore_pressure
        ORDER BY RANDOM()
        LIMIT 3
    """).df().to_string(index=False))

    # ---- 12. gas_reading sample ----
    print("\n" + "=" * 70)
    print("12. GAS_READING — 3 sample rows")
    print("=" * 70)
    n_gas = con.execute("SELECT COUNT(*) FROM gas_reading").fetchone()[0]
    if n_gas > 0:
        print(con.execute("""
            SELECT well_family, sample_time, gas_class, depth_top_md, depth_bottom_md
            FROM gas_reading
            ORDER BY RANDOM()
            LIMIT 3
        """).df().to_string(index=False))
    else:
        print("  (no gas readings — most reports have empty gas sections)")

    # ---- 13. parse errors ----
    print("\n" + "=" * 70)
    print("13. PARSE_ERRORS — any failures during ingestion?")
    print("=" * 70)
    n_err = con.execute("SELECT COUNT(*) FROM parse_errors").fetchone()[0]
    if n_err == 0:
        print("  ✅ No parse errors")
    else:
        print(f"  ⚠️  {n_err} errors:")
        print(con.execute("""
            SELECT stage, error_message, COUNT(*) AS n
            FROM parse_errors
            GROUP BY stage, error_message
            ORDER BY n DESC
        """).df().to_string(index=False))

    # ---- 14. cross-table referential sanity check ----
    print("\n" + "=" * 70)
    print("14. REFERENTIAL SANITY — every operation links to a report?")
    print("=" * 70)
    orphans = con.execute("""
        SELECT COUNT(*) FROM operations o
        WHERE NOT EXISTS (SELECT 1 FROM reports r WHERE r.pdf_path = o.pdf_path)
    """).fetchone()[0]
    print(f"  Orphan operations (no matching report): {orphans}")

    pdfs_with_no_ops = con.execute("""
        SELECT COUNT(*) FROM reports r
        WHERE NOT EXISTS (SELECT 1 FROM operations o WHERE o.pdf_path = r.pdf_path)
    """).fetchone()[0]
    print(f"  Reports with no operations: {pdfs_with_no_ops} (expected — placeholder PDFs)")

print("\n" + "=" * 70)
print("✅ DATABASE VERIFICATION COMPLETE")
print("=" * 70)

INFO     NumExpr defaulting to 12 threads.

1. TABLES & ROW COUNTS
  reports                 1,000
  operations             11,199
  drilling_fluid          2,739
  pore_pressure             598
  survey_station            993
  lithology                 200
  gas_reading               290
  parse_errors                0

2. REPORTS — populated count per column (out of total reports)
                      column  populated   pct
                    pdf_path       1000 100.0
                pdf_filename       1000 100.0
                 well_prefix       1000 100.0
                 well_family       1000 100.0
                 wellbore_id       1000 100.0
               report_number       1000 100.0
                 report_date       1000 100.0
                period_start       1000 100.0
                  period_end       1000 100.0
                      status       1000 100.0
        report_creation_time       1000 100.0
           days_ahead_behind        361  36.1
                    operator        999  99.9
             

In [1]:
# Diagnostic: what does pdfplumber actually emit, and why is the header parser missing it?
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.common.config import RAW_PDF_DIR
from src.ingestion.text_extraction import extract_text, normalize_text
from src.ingestion.sections import parse_header

# Pick any modern F-12 PDF
pdf = sorted(RAW_PDF_DIR.glob("15_9_F_12_2008*.pdf"))[0]
print(f"PDF: {pdf.name}\n")

# Step 1: see RAW pdfplumber output (no normalization)
import pdfplumber
with pdfplumber.open(pdf) as pdfobj:
    raw = pdfobj.pages[0].extract_text()
print("=" * 70)
print("STEP 1 — RAW pdfplumber output (first 600 chars):")
print("=" * 70)
print(repr(raw[:600]))

# Step 2: see what extract_text() (with normalization) gives
text = extract_text(pdf)
print("\n" + "=" * 70)
print("STEP 2 — After normalize_text() (first 600 chars):")
print("=" * 70)
print(repr(text[:600]))

# Step 3: try parse_header on it
print("\n" + "=" * 70)
print("STEP 3 — parse_header() result:")
print("=" * 70)
h = parse_header(text)
for k, v in h.items():
    print(f"  {k:35s}: {v!r}")

PDF: 15_9_F_12_2008_01_01.pdf

STEP 1 — RAW pdfplumber output (first 600 chars):
'Summary report\nWWeellllbboorree:: 15/9-F-12 PPeerriioodd:: 2007-12-31 00:00 - 2008-01-01 00:00\nStatus: normal Dist Drilled (m): -999.99 Depth at Kick Off mMD:\nReport creation time: 2018-05-03 13:51 Penetration rate (m/h): -999.99 Depth at Kick Off mTVD:\nReport number: 98 Hole Dia (): Depth mMd: 3520\nDays Ahead/Behind (+/-): 110.6 Pressure Test Type: formation integrity tes Depth mTVD: 3107.4\nt\nOperator: StatoilHydro Plug Back Depth mMD:\nFormation strength (g/cm3): 1.6\nRig Name: MÆRSK INSPIRER Depth at formation strength mMD: 3116\nDia Last Casing ():\nDrilling contractor: Mærsk Contractors Depth'

STEP 2 — After normalize_text() (first 600 chars):
'Summary report\nWellbore: 15/9-F-12 Period: 2007-12-31 00:00 - 2008-01-01 00:00\nStatus: normal Dist Drilled (m): -999.99 Depth at Kick Off mMD:\nReport creation time: 2018-05-03 13:51 Penetration rate (m/h): -999.99 Depth at Kick Off mTVD:\nReport num

In [5]:
# Find 26" BHA mentions across all F-12
with get_connection(read_only=True) as con:
    df_26 = con.execute("""
        SELECT pdf_path, start_time, SUBSTRING(remark, 1, 250) AS remark
        FROM operations
        WHERE well_family = '15_9_F_12'
          AND (
            remark LIKE '%26"%'
            OR remark LIKE '%26''%'
            OR remark LIKE '% 26 %'
          )
          AND remark ILIKE '%BHA%'
        ORDER BY pdf_path
        LIMIT 20
    """).df()
print(f"=== F-12 ops mentioning 26\" + BHA: {len(df_26)} ===")
display(df_26)

=== F-12 ops mentioning 26" + BHA: 20 ===


,pdf_path,start_time,remark
0,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_21.pdf,03:30,"Held handover meeting on drillfloor. Held toolbox talk prior to picking up 26"" drillout BHA."
1,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_21.pdf,04:00,"Picked up and installed 12 1/4"" x 17 1/2"" x 26"" drillout BHA. Gauged bit. Meanwhile worked to function flowline isolation valve."
2,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_21.pdf,06:00,"Continued making up 26"" drillout BHA. RIH with 26"" drillout BHA to 180 m MD. Meanwhile worked to function flowline isolation valve. 09:00:00 180 interruption -- other ok Worked to function flowlin..."
3,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_22.pdf,14:30,"RIH with 26"" hole opner BHA from 180 m to 222 m. Washed down from 222 m with 2000 lpm, 12 bar. Tagged cement at 250 m with 3 ton. Up weight 94 t/ Down weight 97 t."
4,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_22.pdf,15:00,"Drilled cmt/shoe track with 26"" hole opner BHA; 3000 lpm, 50 rpm and 1-3 ton wob. Picked bit off bottom and stopped pumps."
5,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_22.pdf,22:30,"Drilled 2 m new formation with 26"" hole opner BHA; 3000 lpm, 55 rpm and 1 ton wob. Reamed rate hole and shoe track. Equipment Failure Information SSttaarrtt ttiimmee DDeepptthh mmMMDD DDeepptthh m..."
6,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_23.pdf,00:00,"Drilled 1,5 m new formation with 26"" hole opner BHA; 3000 lpm, 55 rpm and 0.5-1 ton wob. Reamed rat hole. Pumped 15 m3 1.12 sg hi-vis pill and circulated out same. M eanwhile held tool box meeting..."
7,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_23.pdf,03:00,"Continued POOH with 26"" HO BHA and racked back same."
8,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_23.pdf,05:30,"Continued POOH with 26"" HO BHA and racked back same."
9,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_23.pdf,06:00,"POOH with 26"" HO BHA from 29 m to 8 m and racked back same. Held tool box meeting prior to POOH with 12 1/4"" x 17 1/2"" x 26"" hole opener assembly. Removed mast er bushing and LO 26"" HO BHA. Cleare..."


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import pandas as pd
from src.common.config import RAW_PDF_DIR, DB_PATH, NDS_EVENTS_XLSX
from src.common.db import get_connection
from src.ingestion.parse_pdf import parse_pdf
from src.ingestion.text_extraction import extract_text

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Pick any PDF and see its raw extracted text

Useful when you suspect the parser is missing something — see exactly what pdfplumber gave you.

In [4]:
# Replace with the filename you want to inspect
pdf_name = "15_9_F_10_2008_11_30.pdf"   # example

candidates = list(RAW_PDF_DIR.glob(pdf_name))
if not candidates:
    candidates = sorted(RAW_PDF_DIR.glob("*.pdf"))[:1]
    print(f"⚠️  {pdf_name} not found, falling back to {candidates[0].name}")

pdf_path = candidates[0]
text = extract_text(pdf_path)
print(f"=== {pdf_path.name} ===\n")
print(text[:3000])

⚠️  15_9_F_10_2008_11_30.pdf not found, falling back to 15_9_19_A_1980_01_01.pdf
=== 15_9_19_A_1980_01_01.pdf ===

Summary report
Wellbore: 15/9-19 A Period: 1979-12-31 00:00 - 1980-01-01 00:00
Status: normal Dist Drilled (m): -999.99 Depth at Kick Off mMD: 2178
Report creation time: 2018-05-03 Penetration rate (m/h): -999.99 Depth at Kick Off mTVD:
13:53
Hole Dia (): Depth mMd: -999.99
Report number: 1
Pressure Test Type: Depth mTVD:
Days Ahead/Behind (+/-):
Formation strength (): Plug Back Depth mMD:
Operator:
Dia Last Casing (): Depth at formation strength mMD:
Rig Name:
Depth At Formation Strength mTVD:
Drilling contractor:
Depth At Last Casing mMD:
Spud Date: 1997-07-25
00:00 Depth At Last Casing mTVD:
Wellbore type:
Elevation RKB-MSL ():
Water depth MSL (m): 84
Tight well: Y
HPHT: Y
Temperature ():
Pressure ():
Date Well Complete: 1997-08-30
Summary of activities (24 Hours)
None
Summary of planned activities (24 Hours)
None
Survey Station
Depth mMD Depth mTVD Inclination ((dega))

## Parse one PDF and explore the dict

In [5]:
parsed = parse_pdf(pdf_path)
print("Top-level keys:", list(parsed.keys()))
print(f"\nparse_quality: {parsed['report']['parse_quality']}")
print(f"operations:    {len(parsed['operations'])} rows")
print(f"fluid samples: {len(parsed['drilling_fluid'])}")

Top-level keys: ['report', 'operations', 'drilling_fluid', 'pore_pressure', 'survey_station', 'lithology', 'gas_reading']

parse_quality: partial
operations:    0 rows
fluid samples: 0


## Run any DuckDB query

In [6]:
with get_connection(read_only=True) as con:
    df = con.execute("""
        SELECT *
        FROM operations
        WHERE remark ILIKE '%stuck%'
          AND well_family = '15_9_F_12'
        LIMIT 10
    """).df()
df

,op_id,pdf_path,well_family,well_prefix,report_date,op_index,start_time,end_time,end_depth_md,main_activity,sub_activity,state,remark,op_text
0,6054,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_06_22.pdf,15_9_F_12,15_9_F_12,2007-06-22,2,14:00,14:30,180.0,drilling,drill,ok,Held tool box meeting prior to drilling out shoe track. Focused on potential for stuck pipe.,drilling | drill | ok | Held tool box meeting prior to drilling out shoe track. Focused on potential for stuck pipe.
1,6929,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2007_09_10.pdf,15_9_F_12,15_9_F_12,2007-09-10,7,07:00,11:30,0.0,drilling,bop/wellh,ok,Attempted to pull diverter - nogo - stuck in diverter housing. Removed lid to access lock dogs. Inspected locks and ...,drilling | bop/wellh | ok | Attempted to pull diverter - nogo - stuck in diverter housing. Removed lid to access loc...
2,7330,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2008_01_29.pdf,15_9_F_12,15_9_F_12,2008-01-29,2,03:00,03:30,2528.0,interruption,rep,ok,DHSV got stuck in lower guiding plate on FMS while lowering DHSV assembly through slips. Managed to free slips. air,interruption | rep | ok | DHSV got stuck in lower guiding plate on FMS while lowering DHSV assembly through slips. M...
3,7942,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2016_08_26.pdf,15_9_F_12,15_9_F_12,2016-08-26,9,10:30,10:45,0.0,plug abandon,equipment recov,ok,"Recovered last joint and discovered tapered mill stuck in the cut, photographed and secured prior to laying out ery","plug abandon | equipment recov | ok | Recovered last joint and discovered tapered mill stuck in the cut, photographe..."
4,8025,C:\Users\user\Downloads\eilink_ra_ds\eilink_ra_ds\data\raw_pdfs\15_9_F_12_2016_08_29.pdf,15_9_F_12,15_9_F_12,2016-08-29,16,13:30,14:30,0.0,plug abandon,other,ok,Loosened bolts on HP riser connector. 2 bolts stuck.,plug abandon | other | ok | Loosened bolts on HP riser connector. 2 bolts stuck.


## Read the NDS events spreadsheet

In [7]:
nds = pd.read_excel(NDS_EVENTS_XLSX)
nds

,Well,Event
0,15/9-F-10,inclination angle was higher than expected. Reduced inclination to to 0.16 deg
1,15/9-F-11,tight hole event was ecountered while drilling interval at 958 m
2,15/9-F-12,"Excessive clay amount ccumulation is observed in BHA while POOH 26"""" BHA"
3,15/9-F-13,"while RIH with 20"" casing there was a differential stuck. The mud was replaced to seawater and 300 m3 of seawater wa..."
